# Capítulo 3 · Actividad: análisis de datos con NumPy y pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/01_actividad_numpy_pandas.ipynb)

En esta actividad haces de **analista de datos**: vas a responder preguntas sobre datos reales usando solo **NumPy** y **pandas**, sin gráficas. Lo importante no es solo que el código funcione, sino que **sepas interpretar los resultados**.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia. Si editas el original tendrás conflictos al sincronizar tu fork.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```
3. Ten a mano las notebooks de clase: `notebooks/lectures/chapter_03/01_numpy.ipynb` y `02_pandas.ipynb`.

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Puedes añadir todas las celdas que necesites.
- En las celdas ✍️ responde con tus palabras y **apoya tus respuestas con números** de tus resultados.
- Las 💡 **pistas** te recuerdan funciones. Si una función no salió en clase, la pista explica cómo usarla.
- Al terminar, ejecuta **Kernel → Restart & Run All** y comprueba que todo funciona de arriba abajo.

| Parte | Datos | Herramienta |
|---|---|---|
| 1. La cafetería | ventas de un mes (generadas en la notebook) | NumPy |
| 2. Las propinas | `data/tips.csv` | pandas (+ un poco de NumPy) |
| Reto opcional | `data/titanic.csv` | pandas |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

---
## Parte 1 · La cafetería ☕ (NumPy)

Una cafetería vende 5 productos y ha registrado cuántas unidades vendió de cada uno durante **4 semanas** (28 días, de lunes a domingo). La dueña quiere saber cuánto gana, qué días van mejor y qué producto le conviene más.

Ejecuta la siguiente celda para generar los datos. **No hace falta que la modifiques ni que entiendas cada línea.**

In [ ]:
# Datos de la cafetería (no hace falta modificar esta celda)
rng = np.random.default_rng(42)


In [ ]:

productos = np.array(["café", "latte", "té", "croissant", "sándwich"])
precios = np.array([1.5, 2.8, 1.2, 2.0, 4.5])       # precio de cada producto, en $

# Unidades vendidas cada día: los fines de semana se vende un 50% más
dia_semana = np.tile(np.arange(7), 4)                # 0 = lunes, ..., 6 = domingo
factor = np.where(dia_semana >= 5, 1.5, 1.0)
promedio = np.outer(factor, [30, 22, 12, 18, 9])     # ventas esperadas por día y producto
ventas_planas = rng.poisson(promedio).ravel()         # ¡todo en una sola fila!

print(ventas_planas[:10], "...")
print("Número de valores:", ventas_planas.size)

### Ejercicio 1 · Dar forma a los datos

`ventas_planas` tiene 140 números seguidos: primero las ventas de los 5 productos del día 1, después las del día 2, y así sucesivamente.

a) Crea una matriz `ventas` con forma `(28, 5)`: una **fila por día** y una **columna por producto** (en el mismo orden que `productos`).

b) Imprime su `shape`, `ndim`, `size` y `dtype`.

c) ¿Cuántos cafés se vendieron el primer día? Comprueba que coincide con `ventas_planas[0]`.

> 💡 **Pista:** `reshape`.

In [ ]:
# Tu código aquí
ventas = ventas_planas.reshape (28,5)

print ("b)shape", ventas.shape)
print ("ndim", ventas.ndim)
print ("size", ventas.size)
print ("dtype", ventas.dtype)

print ("c)Ventas del dia 1:", ventas[0,0])
print (ventas_planas [0])

### Ejercicio 2 · Indexación

Obtén, usando índices y *slicing* sobre `ventas`:

a) Las ventas de todos los productos el **primer día**.

b) Las ventas de **latte** (columna 1) durante todo el mes.

c) Las ventas de la **última semana** (los últimos 7 días). Comprueba que su forma es `(7, 5)`.

d) Las ventas de **croissant y sándwich** (las dos últimas columnas) durante la **primera semana**.

> 💡 **Pista:** `M[filas, columnas]`. Recuerda que los índices negativos cuentan desde el final.

In [ ]:
# Tu código aquí
ventas_primer_dia = ventas [0]
print("a) Ventas de todos los productos del primer dia:", ventas_primer_dia)
ventas_latte = ventas [:, 1]
print("b) ventas de latte de todo el mes", ventas_latte)
ventas_ultima_semana = ventas [-7 :]
print("c) ventas de la última semana:\n", ventas_ultima_semana)
print(" Shape:" ,ventas_ultima_semana.shape)
ventas_croi_san_semana1 = ventas [:7, -2:]
print ("d) ventas de croissant y sándwich la primer semana:\n", ventas_croi_san_semana1)

### Ejercicio 3 · Ingresos

a) Calcula la matriz `ingresos` (forma `(28, 5)`): lo que se ingresó por cada producto cada día.

b) ¿Cuánto ingresó la cafetería en todo el mes?

c) Crea `ingreso_diario`, un array con el ingreso total de **cada día** (forma `(28,)`).

d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?

> 💡 **Pistas:**
> - `ventas * precios` multiplica **cada fila** de `ventas` por el vector `precios`, elemento a elemento: la columna del café por 1.5, la del latte por 2.8… (este mecanismo se llama *broadcasting*).
> - `M.sum()` suma todo. `M.sum(axis=1)` suma **cada fila** y devuelve un resultado por fila; `M.sum(axis=0)` suma **cada columna**.

In [ ]:
# Tu código aquí
# a) Matriz de ingresos (28, 5)
ingresos = ventas * precios

# b) Ingreso total del mes
ingreso_total = ingresos.sum()
print("Ingreso total del mes:", ingreso_total)

# c) Ingreso diario con forma MATRICIAL (28, 1)
ingreso_diario = ingresos.sum(axis=1, keepdims=True)

print("Matriz de ingreso diario:\n", ingreso_diario)
print("Forma (shape):", ingreso_diario.shape)  # Imprimirá (28, 1)

# d) Ingreso diario medio, máximo y mínimo
medio = ingreso_diario.mean()
maximo = ingreso_diario.max()
minimo = ingreso_diario.min()

print("\nIngreso diario medio:", medio)
print("Ingreso diario máximo:", maximo)
print("Ingreso diario mínimo:", minimo)

### Ejercicio 4 · Días buenos y días flojos

Usa este array con el número de cada día (del 1 al 28):
```python
dias = np.arange(1, 29)
```

a) Crea una máscara con los días cuyo ingreso supera el **ingreso diario medio** y muestra **qué días** fueron.

b) ¿**Cuántos** días fueron?

c) Crea un array que diga `"bueno"` o `"flojo"` para cada día, según si supera o no la media.

d) Calcula `dias[mascara] % 7` (el resto de dividir entre 7). El día 1 fue lunes, así que un resto de **6** es sábado y un resto de **0** es domingo. ¿Qué tienen en común los días buenos?

> 💡 **Pistas:** `.sum()` sobre una máscara cuenta los `True`, porque `True` vale 1 y `False` vale 0. Para c), `np.where`.

In [ ]:
# Tu código aquíimport shutil

# Definición del arreglo de días (del 1 al 28)
dias = np.arange(1, 29)

# a) Máscara booleana y días que superan el ingreso diario medio
# .flatten() asegura que la máscara sea un vector 1D si ingreso_diario era matricial
mascara = (ingreso_diario > ingreso_diario.mean()).flatten()
dias_buenos = dias[mascara]

print("a) Días que superan la media:", dias_buenos)

# b) Cantidad de días buenos
num_dias_buenos = mascara.sum()
print("b) Total de días buenos:", num_dias_buenos)

# c) Array con "bueno" o "flojo"
clasificacion = np.where(mascara, "bueno", "flojo")
print("c) Clasificación por día:\n", clasificacion)

# d) Resto de dividir los días buenos entre 7
restos = dias[mascara] % 7
print("d) Resto (% 7) de los días buenos:", restos)

✍️ **Tu respuesta:** 

### Ejercicio 5 · Análisis por semanas

a) Reorganiza `ingreso_diario` en una matriz `por_semana` de forma `(4, 7)`: una fila por semana y una columna por día (lunes a domingo).

b) ¿Cuánto se ingresó **cada semana**?

c) ¿Cuál es el ingreso medio de **cada día de la semana**? ¿Qué día se vende más? Usa:
```python
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
```

d) Compara el ingreso medio de un día de **fin de semana** con el de un día **entre semana**. ¿Cuántas veces más se ingresa en fin de semana?

> 💡 **Pistas:** `axis=1` da un resultado por fila (por semana) y `axis=0` uno por columna (por día de la semana). `arr.argmax()` devuelve la **posición** del valor máximo; úsala como índice en `nombres_dias`.

In [ ]:
# Tu código aquí
# Nombres de los días
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])

# a) Reorganizar ingreso_diario en una matriz (4, 7)
por_semana = ingreso_diario.reshape(4, 7)
print("a) Matriz por semana (forma):", por_semana.shape)

# b) Ingreso por cada semana (suma a lo largo del eje 1 / filas)
ingreso_cada_semana = por_semana.sum(axis=1)
print("\nb) Ingreso total de cada semana:", ingreso_cada_semana)

# c) Ingreso medio de cada día de la semana y el día de mayores ventas (promedio a lo largo del eje 0 / columnas)
media_por_dia = por_semana.mean(axis=0)
posicion_max = media_por_dia.argmax()
dia_max = nombres_dias[posicion_max]

print("\nc) Ingreso medio por día de la semana:")
for dia, media in zip(nombres_dias, media_por_dia):
    print(f"   {dia.capitalize()}: {media:.2f}")

print(f"   -> El día que más se vende es el **{dia_max}**.")

# d) Comparación entre semana (lunes-viernes, cols 0 a 4) y fin de semana (sábado-domingo, cols 5 y 6)
media_entre_semana = por_semana[:, :5].mean()  # O media_por_dia[:5].mean()
media_fin_de_semana = por_semana[:, 5:].mean() # O media_por_dia[5:].mean()

veces_mas = media_fin_de_semana / media_entre_semana

print(f"\nd) Ingreso medio entre semana: {media_entre_semana:.2f}")
print(f"   Ingreso medio en fin de semana: {media_fin_de_semana:.2f}")
print(f"   -> En fin de semana se ingresa **{veces_mas:.2f} veces más** que entre semana.")

### Ejercicio 6 · El producto estrella

a) Calcula el ingreso total de **cada producto** en el mes.

b) ¿Qué **porcentaje** del ingreso total aporta cada producto? Redondea a 1 decimal.

c) ¿Qué producto genera **más dinero**? ¿Y cuál vende **más unidades**?

> 💡 **Pista:** `axis=0` y `argmax`, igual que en el ejercicio anterior.

In [ ]:
# Tu código aquí
# a) Ingreso total de cada producto en el mes (suma por columnas / axis=0)
ingreso_por_producto = ingresos.sum(axis=0)

print("a) Ingreso total por producto:")
for prod, ing in zip(productos, ingreso_por_producto):
    print(f"   {prod}: {ing:.2f}")

# b) Porcentaje del ingreso total que aporta cada producto (redondeado a 1 decimal)
porcentaje_ingresos = np.round((ingreso_por_producto / ingresos.sum()) * 100, 1)

print("\nb) Porcentaje del ingreso total por producto:")
for prod, porc in zip(productos, porcentaje_ingresos):
    print(f"   {prod}: {porc}%")

# c) Producto que genera más dinero y producto que vende más unidades
# Más dinero (sobre la matriz ingresos)
idx_mas_dinero = ingreso_por_producto.argmax()
prod_mas_dinero = productos[idx_mas_dinero]

# Más unidades (sobre la matriz ventas)
unidades_por_producto = ventas.sum(axis=0)
idx_mas_unidades = unidades_por_producto.argmax()
prod_mas_unidades = productos[idx_mas_unidades]

print(f"\nc) Producto que genera MÁS DINERO: {prod_mas_dinero} ({ingreso_por_producto[idx_mas_dinero]:.2f})")
print(f"   Producto que vende MÁS UNIDADES: {prod_mas_unidades} ({unidades_por_producto[idx_mas_unidades]} unidades)")

✍️ **Pregunta:** ¿por qué el producto que más se vende no es el que más dinero deja? Si fueras la dueña, ¿qué harías con esta información?

✍️ **Tu respuesta:** 

---
## Parte 2 · Las propinas 💵 (pandas)

Un camarero anotó, durante varias semanas, cada mesa que atendió: cuánto pagaron, cuánta propina dejaron y algunos datos de la mesa. Queremos responder una pregunta: **¿de qué depende la propina?**

El archivo es `data/tips.csv` y tiene estas columnas:

| Columna | Significado |
|---|---|
| `total_bill` | importe de la cuenta, en $ |
| `tip` | propina, en $ |
| `sex` | sexo de quien pagó (`Female` / `Male`) |
| `smoker` | si en la mesa había fumadores (`Yes` / `No`) |
| `day` | día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | `Lunch` (comida) o `Dinner` (cena) |
| `size` | número de personas en la mesa |

### Ejercicio 7 · Primer vistazo

a) Carga el archivo en un DataFrame llamado `propinas`.

b) Explóralo con `shape`, `head`, `info` y `describe`.

c) ¿Cuántas mesas hay de cada día? ¿Y de cada momento (comida/cena)?

In [ ]:
for archivo in DATA_DIR.glob("*.csv"):
    print(archivo.name)

In [ ]:
propinas = pd.read_csv("../data/tips.csv")

In [ ]:
propinas.head()

In [ ]:
# 1. Llamar / cargar el CSV
propinas = pd.read_csv("../data/tips.csv")

# 2. Visualizar la tabla
propinas.head()

In [ ]:
# Tu código 
# a) Cargar el archivo
propinas = pd.read_csv("../data/tips.csv")

# b) Explorar
print("Forma:", propinas.shape)
display(propinas.head())
propinas.info()
display(propinas.describe())

# c) Conteo de mesas
print("\nMesas por día:")
print(propinas["day"].value_counts())

print("\nMesas por momento:")
print(propinas["time"].value_counts())

✍️ **Pregunta:** ¿cuántas mesas hay? ¿Falta algún dato? ¿Cuál fue la cuenta más alta y cuál la mesa más grande?

✍️ **Tu respuesta:** 
*244 Mesas*
*No falta ningún dato*
*La cuenta más alta fe de 50.81*
*La mesa más grande tuvo 6 comensales*

### Ejercicio 8 · Traducir los datos

Para trabajar más cómodos, vamos a pasar todo a español.

a) Renombra las columnas: `total_bill` → `cuenta`, `tip` → `propina`, `sex` → `sexo`, `smoker` → `fumador`, `day` → `dia`, `time` → `momento`, `size` → `personas`.

b) Traduce los valores:
- `dia`: `Thur` → `Jueves`, `Fri` → `Viernes`, `Sat` → `Sábado`, `Sun` → `Domingo`
- `momento`: `Lunch` → `Comida`, `Dinner` → `Cena`
- `sexo`: `Female` → `Mujer`, `Male` → `Hombre`
- `fumador`: `Yes` → `sí`, `No` → `no`

Comprueba el resultado con `head()`.

> 💡 **Pista:** `rename(columns={...})` y `.map({...})`.

In [ ]:
# Tu código aquí
# a) Renombrar las columnas
columnas_espanol = {
    "total_bill": "cuenta",
    "tip": "propina",
    "sex": "sexo",
    "smoker": "fumador",
    "day": "dia",
    "time": "momento",
    "size": "personas"
}

propinas = propinas.rename(columns=columnas_espanol)

# b) Traducir los valores de cada columna
propinas["dia"] = propinas["dia"].map({
    "Thur": "Jueves",
    "Fri": "Viernes",
    "Sat": "Sábado",
    "Sun": "Domingo"
})

propinas["momento"] = propinas["momento"].map({
    "Lunch": "Comida",
    "Dinner": "Cena"
})

propinas["sexo"] = propinas["sexo"].map({
    "Female": "Mujer",
    "Male": "Hombre"
})

propinas["fumador"] = propinas["fumador"].map({
    "Yes": "sí",
    "No": "no"
})

# Comprobar el resultado
propinas.head()

### Ejercicio 9 · Nuevas columnas

Una propina de 5 $ no significa lo mismo en una cuenta de 15 $ que en una de 50 $. Por eso vamos a crear variables más útiles:

a) `pct_propina`: la propina como **porcentaje** de la cuenta.

b) `cuenta_por_persona`: lo que pagó cada persona de la mesa.

c) `tamaño_mesa`: agrupa `personas` en `"1-2"`, `"3-4"` y `"5-6"`.

d) `generosa`: `"sí"` si la propina es del 20% o más, `"no"` en caso contrario.

> 💡 **Pistas:** operaciones entre columnas, `pd.cut` (con `bins=[0, 2, 4, 6]`) y `np.where`.

In [ ]:
# Tu código aquí

# a) Porcentaje de propina respecto a la cuenta
propinas["pct_propina"] = (propinas["propina"] / propinas["cuenta"]) * 100

# b) Lo que pagó cada persona
propinas["cuenta_por_persona"] = propinas["cuenta"] / propinas["personas"]

# c) Agrupar tamaño de mesa en categorías
propinas["tamaño_mesa"] = pd.cut(
    propinas["personas"], 
    bins=[0, 2, 4, 6], 
    labels=["1-2", "3-4", "5-6"]
)

# d) ¿Propina generosa? (20% o más)
propinas["generosa"] = np.where(propinas["pct_propina"] >= 20, "sí", "no")

# Comprobar las nuevas columnas añadidas
propinas.head()

### Ejercicio 10 · Filtrar y ordenar

a) ¿Cuántas mesas dejaron una propina **generosa** (20% o más)? ¿Qué **proporción** del total representan?

b) ¿Cuántas **cenas de fin de semana** (sábado o domingo) fueron de **4 personas o más**? ¿Cuál fue su porcentaje de propina medio?

c) Muestra las **5 mesas** con mayor `pct_propina`, con las columnas `cuenta`, `propina`, `pct_propina`, `personas` y `dia`.

d) Usa `query` para obtener las mesas con **fumadores** y una cuenta **entre 10 y 20 $**. ¿Cuántas son?

> 💡 **Pistas:** filtros con `&`, `isin` y `sort_values`. Dentro de `query` el texto va entre comillas simples: `"fumador == 'sí'"`.

In [ ]:
# Tu código aquí

# a) Mesas generosas y proporción del total
mesas_generosas = (propinas["generosa"] == "sí").sum()
proporcion_generosas = (propinas["generosa"] == "sí").mean()

print(f"a) Mesas generosas: {mesas_generosas}")
print(f"   Proporción del total: {proporcion_generosas:.3f} ({proporcion_generosas * 100:.1f}%)")

# b) Cenas de fin de semana (sábado/domingo) de 4 personas o más y su media

filtro_cenas = (
    (propinas["momento"] == "Cena") & 
    (propinas["dia"].isin(["Sábado", "Domingo"])) & 
    (propinas["personas"] >= 4)
)

cenas_finde_grandes = propinas[filtro_cenas]
media_propina_cenas = cenas_finde_grandes["pct_propina"].mean()

print(f"\nb) Cenas de fin de semana con 4+ personas: {len(cenas_finde_grandes)}")
print(f"   Porcentaje medio de propina: {media_propina_cenas:.2f}%")

# c) Top 5 mesas con mayor pct_propina (columnas elegidas)
columnas_top = ["cuenta", "propina", "pct_propina", "personas", "dia"]
top_5_propinas = (
    propinas
    .sort_values("pct_propina", ascending=False)
    [columnas_top]
    .head(5)
)

print("\nc) Las 5 mesas con mayor porcentaje de propina:")
display(top_5_propinas)

# d) query: mesas con fumadores y cuenta entre 10 y 20 $
mesas_fumadores_10_20 = propinas.query("fumador == 'sí' and 10 <= cuenta <= 20")

print(f"d) Mesas con fumadores y cuenta entre 10 y 20 $: {len(mesas_fumadores_10_20)}")

✍️ **Pregunta:** fíjate en la primera mesa del apartado c). ¿Te parece un caso normal? ¿Cómo crees que afecta a la **media** del porcentaje de propina?

✍️ **Tu respuesta:** 

### Ejercicio 11 · Agrupar y resumir

a) Para cada **día**, calcula el número de mesas, la **media** y la **mediana** de `pct_propina`.

b) Para cada **momento** (comida/cena), crea un resumen con columnas con nombre: `n_mesas`, `cuenta_media`, `propina_media` y `pct_medio`.

c) Para cada número de **personas**, calcula el número de mesas y el `pct_propina` medio.

d) Crea una tabla dinámica con el `pct_propina` medio por **sexo** (filas) y **fumador** (columnas).

e) Quédate solo con las mesas del **domingo** y calcula la media y la mediana de `pct_propina` **con** y **sin** la mesa del 71% que encontraste en el ejercicio 10.

> 💡 **Pistas:** `groupby(...).agg([...])`, `agg(nombre=("columna", "función"))` y `pd.pivot_table`. Redondea con `.round(1)` para que se lea mejor.

In [ ]:
# Tu código aquí
# a) Por día: número de mesas, media y mediana de pct_propina
resumen_dia = (
    propinas
    .groupby("dia")["pct_propina"]
    .agg(["count", "mean", "median"])
    .round(1)
)
print("a) Resumen por día:")
display(resumen_dia)

In [ ]:
# Tu código aquí
# b) Por momento: n_mesas, cuenta_media, propina_media y pct_medio
resumen_momento = (
    propinas
    .groupby("momento")
    .agg(
        n_mesas=("cuenta", "count"),
        cuenta_media=("cuenta", "mean"),
        propina_media=("propina", "mean"),
        pct_medio=("pct_propina", "mean")
    )
    .round(1)
)
print("\nb) Resumen por momento:")
display(resumen_momento)

In [ ]:
# Tu código aquí
# c) Por número de personas: número de mesas y pct_propina medio
resumen_personas = (
    propinas
    .groupby("personas")
    .agg(
        n_mesas=("cuenta", "count"),
        pct_medio=("pct_propina", "mean")
    )
    .round(1)
)
print("\nc) Resumen por personas en la mesa:")
display(resumen_personas)

In [ ]:
# Tu código aquí
# d) Tabla dinámica: pct_propina medio por sexo (filas) y fumador (columnas)
tabla_dinamica = pd.pivot_table(
    propinas,
    values="pct_propina",
    index="sexo",
    columns="fumador",
    aggfunc="mean"
).round(1)
print("\nd) Tabla dinámica (pct_propina medio):")
display(tabla_dinamica)


In [ ]:
# Tu código aquí
# e) Mesas del domingo: media y mediana con y sin el outlier del 71%
mesas_domingo = propinas[propinas["dia"] == "Domingo"]
# Con la mesa del 71%
media_con = mesas_domingo["pct_propina"].mean()
mediana_con = mesas_domingo["pct_propina"].median()
# Sin la mesa del 71% (filtrando valores menores al 70%)
mesas_domingo_sin = mesas_domingo[mesas_domingo["pct_propina"] < 70]
media_sin = mesas_domingo_sin["pct_propina"].mean()
mediana_sin = mesas_domingo_sin["pct_propina"].median()
print("\ne) Análisis del domingo (efecto de la mesa del 71%):")
print(f"   CON la mesa del 71% -> Media: {media_con:.2f}% | Mediana: {mediana_con:.2f}%")
print(f"   SIN la mesa del 71% -> Media: {media_sin:.2f}% | Mediana: {mediana_sin:.2f}%")

✍️ **Preguntas:**
1. En a), ¿en qué días la media es mayor que la mediana? Según e), ¿cuál de las dos medidas se ve más afectada por una mesa extrema? ¿Cuál usarías para describir la propina "típica"?
2. En c), ¿las mesas más grandes dejan más o menos porcentaje de propina? ¿Te fiarías de los resultados para 1, 5 o 6 personas? ¿Por qué?

✍️ **Tu respuesta:** 

### Ejercicio 12 · Cuidado con las comparaciones

Haz una tabla cruzada que cuente cuántas mesas hay de cada **día** (filas) en cada **momento** (columnas).

> 💡 **Pista:** `pd.crosstab`.

In [ ]:
# Tu código aquí

✍️ **Pregunta:** si comparas las propinas de la comida con las de la cena, ¿estás comparando solo el momento del día, o también otra cosa? ¿Por qué es importante tenerlo en cuenta antes de sacar conclusiones?

✍️ **Tu respuesta:** 

### Ejercicio 13 · Combinar tablas

Tenemos otra tabla con información de cada día:
```python
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})
```

a) Añade esta información a `propinas` con `merge`. Comprueba que el número de filas **no cambia**.

b) Compara la **cuenta media** y el **`pct_propina` medio** entre fin de semana y entre semana.

c) Calcula cuánto se facturó en total **cada día**, ordenado cronológicamente (de jueves a domingo).

> 💡 **Pista:** para c), agrupa por `["orden", "dia"]`: `groupby` ordena los grupos por la primera columna.

In [ ]:
# Tu código aquí

### Ejercicio 14 · De pandas a NumPy

Los modelos de Machine Learning trabajan con arrays de NumPy, así que es habitual pasar de un DataFrame a un array.

a) Crea `X` con las columnas `cuenta`, `propina` y `personas` convertidas a array de NumPy. ¿Qué forma tiene?

b) **Solo con NumPy**, calcula el porcentaje de propina de cada mesa a partir de `X`. Comprueba que su media coincide con la de `propinas["pct_propina"]`.

c) **Solo con NumPy**, ¿cuántas mesas de **2 personas** dejaron **más de 4 $** de propina?

> 💡 **Pistas:** `.to_numpy()`. En `X`, la columna 0 es la cuenta, la 1 la propina y la 2 las personas.

In [ ]:
# Tu código aquí

### ✍️ Conclusiones de la Parte 2

Imagina que el camarero te pregunta: *"¿qué tengo que hacer para conseguir mejores propinas?"*. Escribe:

1. **Tres hallazgos** sobre las propinas, cada uno apoyado en un número concreto de tus resultados.
2. **Una limitación** de estos datos que haga que debas tener cuidado con tus conclusiones.

✍️ **Tu respuesta:** 

---
## Reto opcional · El Titanic 🚢

En 1912 el Titanic se hundió en su viaje inaugural. El archivo `data/titanic.csv` tiene información de 891 pasajeros. Las columnas que necesitas:

| Columna | Significado |
|---|---|
| `survived` | 1 si sobrevivió, 0 si no |
| `sex` | `male` / `female` |
| `age` | edad (¡algunas faltan!) |
| `fare` | precio del billete |
| `class` | clase del billete: `First`, `Second`, `Third` |

**Pregunta:** ¿se cumplió el famoso *"las mujeres y los niños primero"*? ¿Importaba la clase del billete?

a) Carga los datos y usa `info()` para ver qué columnas tienen valores faltantes.

b) ¿Qué proporción de pasajeros sobrevivió?

c) Calcula la tasa de supervivencia por **sexo**, por **clase** y en una tabla dinámica **sexo × clase**.

d) Agrupa las edades en `"niño"` (0–12), `"adolescente"` (13–18), `"adulto"` (19–60) y `"mayor"` (más de 60), y calcula la tasa de supervivencia y el número de pasajeros de cada grupo.

e) ¿Cuánto costaba de media el billete en cada clase?

> 💡 **Pistas:**
> - La **media** de una columna de 0 y 1 es la **proporción** de unos: `titanic["survived"].mean()` es la tasa de supervivencia.
> - Para d), `pd.cut` con `bins=[0, 12, 18, 60, 100]`. Los pasajeros sin edad (`NaN`) quedan fuera de los grupos automáticamente.

In [ ]:
# Tu código aquí
`data/titanic.csv`

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿se cumplió *"las mujeres y los niños primero"*? ¿Y la clase, importaba? Responde con números.

✍️ **Tu respuesta:** 